In [ ]:
import os
import torch
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
import torch.nn.functional as F
from math import log10
from skimage.metrics import structural_similarity as ssim

class LensingSR_Real_Dataset(Dataset):
    def __init__(self, lr_dir, hr_dir, transform=None):
        self.lr_dir = lr_dir
        self.hr_dir = hr_dir
        self.transform = transform
        self.filenames = sorted([f for f in os.listdir(hr_dir) if f.endswith('.npy')])

    def __len__(self): return len(self.filenames)

    def __getitem__(self, idx):
        hr_fname = self.filenames[idx]
        # Dynamically determine the LR filename by replacing the prefix
        lr_fname = hr_fname.replace('HR', 'LR')
        
        # Verify LR file existence; fallback to hr_fname if no prefix-swap is needed
        if not os.path.exists(os.path.join(self.lr_dir, lr_fname)):
            lr_fname = hr_fname

        def robust_load(base_path, filename):
            data = np.load(os.path.join(base_path, filename), allow_pickle=True)
            curr = data
            while True:
                if isinstance(curr, np.ndarray) and curr.dtype == object and curr.ndim == 0:
                    curr = curr.item()
                elif isinstance(curr, (list, tuple)) and len(curr) > 0:
                    curr = curr[0]
                elif isinstance(curr, np.ndarray) and curr.dtype == object and curr.ndim > 0:
                    curr = curr[0]
                else: break
            img = np.ascontiguousarray(curr, dtype=np.float32)
            if img.ndim == 2: img = np.expand_dims(img, axis=0)
            elif img.ndim == 3 and img.shape[0] != 1: img = img[0:1, :, :]
            return torch.from_numpy(img)

        hr_tensor = robust_load(self.hr_dir, hr_fname)
        lr_tensor = robust_load(self.lr_dir, lr_fname)
        
        if self.transform:
            seed = np.random.randint(2147483647)
            torch.manual_seed(seed); hr_tensor = self.transform(hr_tensor)
            torch.manual_seed(seed); lr_tensor = self.transform(lr_tensor)
            
        return lr_tensor, hr_tensor

class EdgeLoss(nn.Module):
    def __init__(self):
        super().__init__()
        # Sobel filters for edge detection
        self.filter_x = torch.tensor([[-1, 0, 1], [-2, 0, 2], [-1, 0, 1]]).float().view(1, 1, 3, 3)
        self.filter_y = torch.tensor([[-1, -2, -1], [0, 0, 0], [1, 2, 1]]).float().view(1, 1, 3, 3)

    def forward(self, sr, hr):
        self.filter_x = self.filter_x.to(sr.device)
        self.filter_y = self.filter_y.to(sr.device)
        
        sr_grad_x = F.conv2d(sr, self.filter_x, padding=1)
        sr_grad_y = F.conv2d(sr, self.filter_y, padding=1)
        hr_grad_x = F.conv2d(hr, self.filter_x, padding=1)
        hr_grad_y = F.conv2d(hr, self.filter_y, padding=1)
        
        return F.l1_loss(sr_grad_x, hr_grad_x) + F.l1_loss(sr_grad_y, hr_grad_y)

criterion_mse = nn.MSELoss()
criterion_edge = EdgeLoss()

real_lr_path = '/kaggle/input/datasets/dundikuladeepeswar/dataset6b/Dataset/LR'
real_hr_path = '/kaggle/input/datasets/dundikuladeepeswar/dataset6b/Dataset/HR'

real_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(90),
])

real_ds = LensingSR_Real_Dataset(real_lr_path, real_hr_path, transform=real_transforms)
real_loader = DataLoader(real_ds, batch_size=16, shuffle=True, num_workers=2, pin_memory=True)

In [16]:
class SwinIR(nn.Module):
    def __init__(self, in_chans=1, embed_dim=96, upscale=2):
        super(SwinIR, self).__init__()
        self.conv_first = nn.Conv2d(in_chans, embed_dim, 3, 1, 1)
        self.upsample = nn.Sequential(
            nn.Conv2d(embed_dim, embed_dim * (upscale ** 2), 3, 1, 1),
            nn.PixelShuffle(upscale),
            nn.Conv2d(embed_dim, in_chans, 3, 1, 1)
        )
    def forward(self, x):
        return self.upsample(self.conv_first(x))

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model_real = SwinIR(upscale=2).to(device)

weights_path = '/kaggle/input/models/dundikuladeepeswar/sr/pytorch/default/1/foundation_sr_finetuned.pth'
state_dict = torch.load(weights_path, map_location=device, weights_only=True)

new_state_dict = {k.replace('module.', ''): v for k, v in state_dict.items()}
model_real.load_state_dict(new_state_dict)

if torch.cuda.device_count() > 1:
    model_real = nn.DataParallel(model_real)


target_model = model_real.module if isinstance(model_real, nn.DataParallel) else model_real
for name, param in target_model.named_parameters():
    if 'conv_first' in name:
        param.requires_grad = False

optimizer_real = optim.AdamW(filter(lambda p: p.requires_grad, model_real.parameters()), lr=1e-5)
criterion_real = nn.MSELoss() 
scaler_real = torch.amp.GradScaler('cuda')

In [18]:
epochs_real = 15
for epoch in range(epochs_real):
    model_real.train()
    epoch_loss = 0
    for lr, hr in real_loader:
        lr, hr = lr.to(device), hr.to(device)
        optimizer_real.zero_grad()
        with torch.amp.autocast('cuda'):
            sr = model_real(lr)
            if sr.shape != hr.shape:
                sr = F.interpolate(sr, size=(hr.shape[2], hr.shape[3]), mode='bilinear')
            loss = criterion_real(sr, hr)
        scaler_real.scale(loss).backward()
        scaler_real.step(optimizer_real)
        scaler_real.update()
        epoch_loss += loss.item()
    print(f"Real-Domain Epoch {epoch+1}/15 | Loss: {epoch_loss/len(real_loader):.6f}")

torch.save(model_real.state_dict(), 'lensing_sr_real_final.pth')

Real-Domain Epoch 1/15 | Loss: 0.005034
Real-Domain Epoch 2/15 | Loss: 0.002975
Real-Domain Epoch 3/15 | Loss: 0.002265
Real-Domain Epoch 4/15 | Loss: 0.001885
Real-Domain Epoch 5/15 | Loss: 0.001741
Real-Domain Epoch 6/15 | Loss: 0.001661
Real-Domain Epoch 7/15 | Loss: 0.001603
Real-Domain Epoch 8/15 | Loss: 0.001580
Real-Domain Epoch 9/15 | Loss: 0.001586
Real-Domain Epoch 10/15 | Loss: 0.001561
Real-Domain Epoch 11/15 | Loss: 0.001582
Real-Domain Epoch 12/15 | Loss: 0.001657
Real-Domain Epoch 13/15 | Loss: 0.001562
Real-Domain Epoch 14/15 | Loss: 0.001548
Real-Domain Epoch 15/15 | Loss: 0.001550


In [19]:
def evaluate_real(model, loader):
    model.eval()
    mse_sum, psnr_sum, ssim_sum = 0, 0, 0
    with torch.no_grad():
        for lr, hr in loader:
            lr, hr = lr.to(device), hr.to(device)
            sr = model(lr).clamp(0, 1)
            if sr.shape != hr.shape:
                sr = F.interpolate(sr, size=(hr.shape[2], hr.shape[3]), mode='bilinear')
            
            mse = F.mse_loss(sr, hr).item()
            mse_sum += mse
            psnr_sum += 10 * log10(1 / (mse + 1e-10))
            
            sr_img = sr[0].cpu().numpy().squeeze()
            hr_img = hr[0].cpu().numpy().squeeze()
            ssim_sum += ssim(sr_img, hr_img, data_range=1)
            
    n = len(loader)
    print(f"Task VI.B FINAL -> MSE: {mse_sum/n:.6f}, PSNR: {psnr_sum/n:.2f}dB, SSIM: {ssim_sum/n:.4f}")

evaluate_real(model_real, real_loader)

Task VI.B FINAL -> MSE: 0.001514, PSNR: 29.00dB, SSIM: 0.8065


In [ ]:
# --- NEW REFINEMENT CELL ---

# 1. Unfreeze all layers for global domain adaptation
for param in model_real.parameters():
    param.requires_grad = True

# 2. Use a much smaller learning rate to avoid destroying pre-trained features
optimizer_final = optim.AdamW(model_real.parameters(), lr=1e-6)

print("Starting Global Refinement with Hybrid Edge-MSE Loss...")

for epoch in range(10):
    model_real.train()
    epoch_loss = 0
    for lr, hr in real_loader:
        lr, hr = lr.to(device), hr.to(device)
        optimizer_final.zero_grad()
        
        with torch.amp.autocast('cuda'):
            sr = model_real(lr)
            if sr.shape != hr.shape:
                sr = F.interpolate(sr, size=hr.shape[2:], mode='bilinear')
            
            # Hybrid Loss: Pixel-wise accuracy + Gradient/Edge sharpness
            loss = 0.8 * criterion_mse(sr, hr) + 0.2 * criterion_edge(sr, hr)
            
        scaler_real.scale(loss).backward()
        scaler_real.step(optimizer_final)
        scaler_real.update()
        epoch_loss += loss.item()
        
    print(f"Refinement Epoch {epoch+1}/10 | Hybrid Loss: {epoch_loss/len(real_loader):.6f}")

print("\n--- FINAL POST-REFINEMENT METRICS ---")
evaluate_real(model_real, real_loader)

torch.save(model_real.state_dict(), 'lensing_sr_real_final_optimized.pth')

Starting Global Refinement with Hybrid Edge-MSE Loss...
Refinement Epoch 1/10 | Hybrid Loss: 0.019783
Refinement Epoch 2/10 | Hybrid Loss: 0.019300
Refinement Epoch 3/10 | Hybrid Loss: 0.019264
Refinement Epoch 4/10 | Hybrid Loss: 0.018971
Refinement Epoch 5/10 | Hybrid Loss: 0.018748
Refinement Epoch 6/10 | Hybrid Loss: 0.018608
Refinement Epoch 7/10 | Hybrid Loss: 0.018308
Refinement Epoch 8/10 | Hybrid Loss: 0.018218
Refinement Epoch 9/10 | Hybrid Loss: 0.017900
Refinement Epoch 10/10 | Hybrid Loss: 0.017647

--- FINAL POST-REFINEMENT METRICS ---
Task VI.B FINAL -> MSE: 0.001503, PSNR: 29.08dB, SSIM: 0.7648
